# Bronze Table Inspection

Ad-hoc scan of `raw_data/bronze/runs`, the Delta table written by the `bronze_runs` Dagster asset (`sts_pipeline/assets/bronze.py`). This is a dev/inspection notebook, not part of the numbered analysis sequence — it exists to eyeball the raw ingested data, not to produce analysis output.

In [ ]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

BRONZE_RUNS_PATH = str(PROJECT_ROOT / "raw_data" / "bronze" / "runs")

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder.master("local[*]")
    .appName("bronze-inspection")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "8g")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(BRONZE_RUNS_PATH)
print(f"Loaded {BRONZE_RUNS_PATH}")

## Schema

Nested columns (`card_choices`, `relics_obtained`, `campfire_choices`, etc.) stay as arrays/structs at bronze rather than being flattened — that reshaping happens at silver.

In [ ]:
df.printSchema()

## Row count and quick sanity checks

Comparable to the checks in `01_data_collection.ipynb` (win rate, floor distribution) — just confirming the Spark ingestion landed the same shape of data as the old pandas pass did.

In [ ]:
print("Row count:", df.count())
print()
print("victory value counts:")
df.groupBy("victory").count().show()
print("character_chosen value counts:")
df.groupBy("character_chosen").count().orderBy("count", ascending=False).show()

## Table view

`.toPandas()` on a small, flat-column slice renders as a normal table in Jupyter. Nested columns are left out here since they'd just show as raw list/struct reprs — see the next section for those.

In [ ]:
flat_cols = [
    "play_id", "character_chosen", "ascension_level", "victory",
    "floor_reached", "score", "build_version", "_source_file",
]
df.select(flat_cols).limit(50).toPandas()

## Peek at a nested column

`card_choices` is the field the whole analysis hinges on — confirm it looks like the `{picked, not_picked, floor}` shape from `01_data_collection.ipynb`.

In [ ]:
sample_row = df.select("play_id", "card_choices").filter(df.card_choices.isNotNull()).first()
print("play_id:", sample_row["play_id"])
for pick in sample_row["card_choices"]:
    print(pick)

## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [ ]:
spark.stop()